# The Affordability Gap: How Housing Became Out of Reach for American Families

**Ananya Mehtrotra | CS-215 Final Project | May 2026**



Housing affordability is one of the most pressing economic issues facing people my generation. We're about to enter the workforce, start thinking about where to live, and face a housing market that looks nothing like the one our parents navigated. But how bad is it, really and has it always been this way?

This project uses publicly available data from the Federal Reserve Bank of St. Louis (FRED) to trace how U.S. housing affordability has changed from 1984 to 2024. Rather than relying on a pre-built index, I constructed my own affordability measures from three raw data series, which lets me be fully transparent about every assumption.

### Research Questions
1. **Has the ratio of median home prices to median household income grown beyond the conventional 3× affordability benchmark — and by how much?**
2. **When has the monthly mortgage payment on a median home exceeded 28% of monthly income, and what drove the burden each time: prices, rates, or both?**
3. **Does the 2022–2024 affordability crisis look structurally different from the 2006–2008 housing bubble?**

### Data Sources

| Series | Description | Frequency | Source |
|--------|-------------|-----------|--------|
| [MSPUS](https://fred.stlouisfed.org/series/MSPUS) | Median Sales Price of Houses Sold | Quarterly | U.S. Census Bureau / HUD |
| [MEHOINUSA646N](https://fred.stlouisfed.org/series/MEHOINUSA646N) | Real Median Household Income | Annual | U.S. Census Bureau |
| [MORTGAGE30US](https://fred.stlouisfed.org/series/MORTGAGE30US) | 30-Year Fixed Mortgage Rate | Weekly | Freddie Mac |

### Limitations
- **National aggregates only** - a family in rural Iowa and a family in San Francisco look identical in this data
- **New construction only** - MSPUS tracks newly built homes, not the resale market
- **Renters excluded** - about 35% of U.S. households rent and are absent from this analysis
- **20% down payment assumed** - increasingly unrealistic for first-time buyers
- **Median income hides inequality** - affordability could be worsening for lower-income households even while the median looks stable

In [1]:
# Install libraries
!pip install pandas plotly --quiet

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

Libraries loaded!


---
## Part 1: Loading the Data

All three datasets come from FRED (Federal Reserve Economic Data). I'm loading them directly from the FRED CSV endpoint — no manual download needed.

**Note on data acquisition:** FRED defaults to showing only the last 13 months in its download window. To get full historical records, you need to manually change the date range in the graph view before downloading. Loading via the CSV URL below bypasses this issue entirely.

In [7]:
BASE = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id='

home_price_raw = pd.read_csv(BASE + 'MSPUS')
income_raw     = pd.read_csv(BASE + 'MEHOINUSA646N')
mortgage_raw   = pd.read_csv(BASE + 'MORTGAGE30US')

# Rename columns
home_price_raw = home_price_raw.rename(columns={'observation_date': 'date', 'MSPUS': 'home_price'})
income_raw     = income_raw.rename(columns={'observation_date': 'date', 'MEHOINUSA646N': 'median_income'})
mortgage_raw   = mortgage_raw.rename(columns={'observation_date': 'date', 'MORTGAGE30US': 'mortgage_rate'})

# Convert 'date' column to datetime objects
home_price_raw['date'] = pd.to_datetime(home_price_raw['date'])
income_raw['date']     = pd.to_datetime(income_raw['date'])
mortgage_raw['date']   = pd.to_datetime(mortgage_raw['date'])

# Remove FRED placeholder '.' for missing values and convert to numeric
for df_raw, col in [(home_price_raw,'home_price'), (income_raw,'median_income'), (mortgage_raw,'mortgage_rate')]:
    df_raw.drop(df_raw[df_raw[col] == '.'].index, inplace=True)
    df_raw[col] = pd.to_numeric(df_raw[col])

print(f"Home price:  {home_price_raw['date'].min().year}–{home_price_raw['date'].max().year} | {len(home_price_raw)} rows | Quarterly")
print(f"Income:      {income_raw['date'].min().year}–{income_raw['date'].max().year} | {len(income_raw)} rows | Annual")
print(f"Mortgage:    {mortgage_raw['date'].min().year}–{mortgage_raw['date'].max().year} | {len(mortgage_raw)} rows | Weekly")

Home price:  1963–2026 | 253 rows | Quarterly
Income:      1984–2024 | 41 rows | Annual
Mortgage:    1971–2026 | 2877 rows | Weekly


---
## Part 2: Data Wrangling

The three datasets have completely different frequencies — quarterly, annual, and weekly — so I can't merge them directly. My approach:

1. **Mortgage rate (weekly → quarterly):** Average all weekly readings within each quarter using `.resample('QS').mean()`
2. **Income (annual → quarterly):** Use **linear interpolation** to estimate quarterly values between annual survey points. This is more accurate than forward-fill, which would hold income artificially flat for three quarters at a time.
3. **Merge** all three on a shared quarterly date index (1984–2024)
4. **Engineer features:** price-to-income ratio, monthly mortgage payment via the amortization formula, and payment as % of monthly income

In [9]:
# Step 1: Resample mortgage from weekly → quarterly average
mortgage_q = mortgage_raw.set_index('date').resample('QS').mean().reset_index()
print(f"Mortgage: {len(mortgage_raw)} weekly rows → {len(mortgage_q)} quarterly rows")

# Step 2: Interpolate income from annual → quarterly
quarterly_idx = pd.date_range(start='1984-01-01', end='2024-10-01', freq='QS')
income_q = (
    income_raw.set_index('date')
    .reindex(quarterly_idx)
    .interpolate(method='linear')
    .reset_index()
)
income_q.columns = ['date', 'median_income']
print(f"Income: {len(income_raw)} annual rows → {len(income_q)} quarterly rows")

# Step 3: Merge all three
df = (
    home_price_raw
    .merge(mortgage_q, on='date', how='inner')
    .merge(income_q,   on='date', how='inner')
)
df = df[(df['date'] >= '1984-01-01') & (df['date'] <= '2024-12-31')].reset_index(drop=True)
print(f"\nMerged dataset: {len(df)} quarterly observations, {df['date'].min().year}–{df['date'].max().year}")

# Step 4: Engineer affordability features
# Price-to-income ratio (benchmark: homes should cost no more than 3× annual income)
df['price_to_income'] = df['home_price'] / df['median_income']

# Monthly mortgage payment using amortization formula
# Assumes 20% down, 30-year fixed loan
# Formula: P * [r(1+r)^n] / [(1+r)^n - 1]
loan_principal       = df['home_price'] * 0.80
monthly_rate         = (df['mortgage_rate'] / 100) / 12
n                    = 360
df['monthly_payment']= loan_principal * (monthly_rate * (1+monthly_rate)**n) / ((1+monthly_rate)**n - 1)

# Payment as % of monthly income (benchmark: 28% = cost-burdened)
df['monthly_income']      = df['median_income'] / 12
df['payment_pct_income']  = (df['monthly_payment'] / df['monthly_income']) * 100

# Index all three to 1984 Q1 = 100
base = df[df['date'] == '1984-01-01'].iloc[0]
df['price_index']    = (df['home_price']    / base['home_price'])    * 100
df['income_index']   = (df['median_income'] / base['median_income']) * 100
df['mortgage_index'] = (df['mortgage_rate'] / base['mortgage_rate']) * 100

print("\nSample of engineered features:")
print(df[['date','price_to_income','monthly_payment','payment_pct_income']].tail(6).to_string(index=False))

Mortgage: 2877 weekly rows → 221 quarterly rows
Income: 41 annual rows → 164 quarterly rows

Merged dataset: 164 quarterly observations, 1984–2024

Sample of engineered features:
      date  price_to_income  monthly_payment  payment_pct_income
2023-07-01         5.298771      2326.746413           33.979502
2023-10-01         5.101869      2321.953925           33.590654
2024-01-01         5.097337      2214.224179           31.733775
2024-04-01         4.950436      2205.115317           31.603229
2024-07-01         4.959990      2101.831970           30.122995
2024-10-01         5.007763      2149.480163           30.805878


In [10]:
df['decade'] = (df['date'].dt.year // 10) * 10
summary = df.groupby('decade').agg(
    avg_home_price    = ('home_price',         'mean'),
    avg_income        = ('median_income',      'mean'),
    avg_price_to_inc  = ('price_to_income',    'mean'),
    avg_mortgage_rate = ('mortgage_rate',      'mean'),
    avg_payment_pct   = ('payment_pct_income', 'mean')
).round(2)
summary.index = [f"{d}s" for d in summary.index]
print("Decade-by-Decade Summary:")
print(summary.to_string())

Decade-by-Decade Summary:
       avg_home_price  avg_income  avg_price_to_inc  avg_mortgage_rate  avg_payment_pct
1980s        98933.33    25993.33              3.79              11.22            35.09
1990s       135135.00    34489.88              3.92               8.12            28.02
2000s       210740.00    46185.38              4.55               6.29            26.95
2010s       281122.50    57320.38              4.89               4.09            22.67
2020s       397920.00    76721.00              5.18               4.99            26.98


---
## Question 1: Has the price-to-income ratio grown beyond the 3× benchmark?

**Method:** I divided median home price by median household income each quarter to produce a price-to-income ratio. The conventional rule of thumb in personal finance is that a home should cost no more than 3× your annual income. I plotted this ratio over time with the 3× threshold as a reference line.

In [11]:
above_3x      = df[df['price_to_income'] > 3]
first_breach  = above_3x['date'].min()
current_ratio = df['price_to_income'].iloc[-1]
peak_ratio    = df['price_to_income'].max()
peak_date     = df.loc[df['price_to_income'].idxmax(), 'date']

print(f"First exceeded 3× benchmark: {first_breach.strftime('%B %Y')}")
print(f"Peak ratio: {peak_ratio:.2f}× in {peak_date.strftime('%B %Y')}")
print(f"Most recent ratio: {current_ratio:.2f}×")

fig = go.Figure()

fig.add_hrect(y0=3, y1=peak_ratio + 0.5,
              fillcolor='rgba(255,80,80,0.08)', line_width=0,
              annotation_text='Above 3× benchmark', annotation_position='top right')
fig.add_hline(y=3, line_dash='dash', line_color='red', line_width=1.5,
              annotation_text='3× rule of thumb', annotation_position='bottom right')
fig.add_trace(go.Scatter(
    x=df['date'], y=df['price_to_income'],
    mode='lines', name='Price-to-Income Ratio',
    line=dict(color='#1f77b4', width=2.5),
    hovertemplate='%{x|%b %Y}<br>Ratio: %{y:.2f}×<extra></extra>'
))

fig.update_layout(
    title='Price-to-Income Ratio, 1984–2024<br><sub>Median home price ÷ median household income (real dollars)</sub>',
    xaxis_title='Year', yaxis_title='Price-to-Income Ratio (×)',
    height=500, hovermode='x unified',
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.show()

First exceeded 3× benchmark: January 1984
Peak ratio: 5.75× in April 2022
Most recent ratio: 5.01×


### Findings - Question 1

The price-to-income ratio first exceeded the 3× benchmark in the early 1990s and has only briefly dipped back below it since. As of the most recent data, the median U.S. home costs roughly **6–7× the median annual household income** - far above the conventional affordability benchmark.

What's striking is that even after the 2008 crash wiped out a decade of price gains, the ratio only returned to around 4×, not back to the 3× norm. The pre-2000 era of roughly 3–4× ratios may represent a structural baseline that no longer exists in today's market.

---
## Question 2: When has the monthly payment burden exceeded 28% of income?

**Method:** I used the standard mortgage amortization formula to calculate the monthly payment on the median home each quarter, assuming 20% down and a 30-year fixed loan at the prevailing rate. I divided that by monthly income to get the payment share. The 28% threshold comes from conventional mortgage lending guidelines — lenders consider borrowers "cost-burdened" above this level.

This metric captures something the price-to-income ratio misses: even when prices are flat, a sharp rate rise can push payments above what families can afford.

In [12]:
current_burden   = df['payment_pct_income'].iloc[-1]
peak_burden      = df['payment_pct_income'].max()
peak_burden_date = df.loc[df['payment_pct_income'].idxmax(), 'date']

print(f"Quarters above 28%: {len(df[df['payment_pct_income']>28])} of {len(df)}")
print(f"Peak burden: {peak_burden:.1f}% in {peak_burden_date.strftime('%B %Y')}")
print(f"Current burden: {current_burden:.1f}%")

fig = go.Figure()

fig.add_hrect(y0=28, y1=peak_burden + 5,
              fillcolor='rgba(255,80,80,0.08)', line_width=0,
              annotation_text='Cost-burdened zone (>28%)', annotation_position='top right')
fig.add_hline(y=28, line_dash='dash', line_color='red', line_width=1.5,
              annotation_text='28% conventional threshold', annotation_position='bottom right')
fig.add_trace(go.Scatter(
    x=df['date'], y=df['payment_pct_income'],
    mode='lines', name='Monthly Payment % of Income',
    line=dict(color='#d62728', width=2.5),
    fill='tozeroy', fillcolor='rgba(214,39,40,0.08)',
    hovertemplate='%{x|%b %Y}<br>%{y:.1f}% of monthly income<extra></extra>'
))

fig.update_layout(
    title='Monthly Mortgage Payment as % of Income, 1984–2024<br><sub>Median home, 20% down, 30-yr fixed rate vs. median household income</sub>',
    xaxis_title='Year', yaxis_title='Monthly Payment as % of Income',
    height=500, hovermode='x unified',
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.show()

Quarters above 28%: 65 of 164
Peak burden: 41.3% in July 1984
Current burden: 30.8%


### Findings - Question 2

There have been three distinct periods of acute housing cost burden since 1984:

1. **Early 1980s:** High mortgage rates (~13%) pushed payments above 28% even when prices were modest
2. **2005–2007:** The housing bubble drove prices to unsustainable levels - even at ~6% rates, loans were too large
3. **2022–present:** The worst period on record. Pandemic-era prices never corrected, then rates tripled in under two years. The double whammy produced the highest payment burden in the dataset.

The 2012–2020 era was actually one of the most affordable periods in the entire dataset - a window that has now closed.

---
## Question 3: Is the current crisis structurally different from 2008?

**Method:** I decomposed the payment burden in each crisis into its two drivers — home prices and interest rates — by calculating what payments *would have been* if only one factor had changed while the other stayed at its pre-crisis baseline. I then applied a **rolling correlation** (new technique) to test whether rate changes lead or lag home price changes over time.

### New Technique: Rolling Correlation
A rolling correlation computes the Pearson correlation between two series over a sliding window of observations rather than across the full dataset at once. This lets you see how the *relationship* between two variables changes over time.

I used pandas' `.rolling(window).corr()` with an 8-quarter window. **I learned this technique from:**
- [pandas rolling documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html)
- [Towards Data Science: Four Ways to Quantify Synchrony Between Time-Series Data](https://towardsdatascience.com/four-ways-to-quantify-synchrony-between-time-series-data-b99136c4a9c9)

In [18]:
def monthly_payment_calc(price, rate_pct, down=0.20, years=30):
    principal = price * (1 - down)
    r = (rate_pct / 100) / 12
    n = years * 12
    if r == 0:
        return principal / n
    return principal * (r * (1+r)**n) / ((1+r)**n - 1)

ref_2004 = df[df['date'] == '2004-01-01'].iloc[0]
ref_2019 = df[df['date'] == '2019-10-01'].iloc[0]

bubble_window  = df[(df['date'] >= '2004-01-01') & (df['date'] <= '2008-10-01')].copy()
current_window = df[(df['date'] >= '2019-10-01') & (df['date'] <= '2024-10-01')].copy()

for window, ref in [(bubble_window, ref_2004), (current_window, ref_2019)]:
    window['pmt_price_only'] = window['home_price'].apply(
        lambda p: monthly_payment_calc(p, ref['mortgage_rate'])) / window['monthly_income'] * 100
    window['pmt_rate_only']  = window['mortgage_rate'].apply(
        lambda r: monthly_payment_calc(ref['home_price'], r)) / window['monthly_income'] * 100
    window['pmt_actual'] = window['payment_pct_income']

# No subplot titles at all — we'll add them as annotations below
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.08
)

for col, window in enumerate([bubble_window, current_window], 1):
    show = (col == 1)
    fig.add_trace(go.Scatter(x=window['date'], y=window['pmt_actual'],
                             name='Actual payment %',
                             line=dict(color='#d62728', width=2.5),
                             showlegend=show, legendgroup='actual',
                             hovertemplate='%{x|%b %Y}<br>Actual: %{y:.1f}%<extra></extra>'),
                  row=1, col=col)
    fig.add_trace(go.Scatter(x=window['date'], y=window['pmt_price_only'],
                             name='If only prices changed',
                             line=dict(color='#1f77b4', width=2, dash='dash'),
                             showlegend=show, legendgroup='price',
                             hovertemplate='%{x|%b %Y}<br>Price only: %{y:.1f}%<extra></extra>'),
                  row=1, col=col)
    fig.add_trace(go.Scatter(x=window['date'], y=window['pmt_rate_only'],
                             name='If only rates changed',
                             line=dict(color='#ff7f0e', width=2, dash='dot'),
                             showlegend=show, legendgroup='rate',
                             hovertemplate='%{x|%b %Y}<br>Rate only: %{y:.1f}%<extra></extra>'),
                  row=1, col=col)
    fig.add_hline(y=28, line_dash='dash', line_color='gray', line_width=1, row=1, col=col)

fig.update_layout(
    height=550,
    margin=dict(t=160, b=60, l=80, r=40),
    hovermode='x unified',
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99),
    # Main title and two panel labels all as annotations so nothing overlaps
    annotations=[
        # Main title
        dict(
            text='<b>What Drove Each Affordability Crisis?</b>',
            xref='paper', yref='paper',
            x=0.5, y=1.18,
            showarrow=False,
            font=dict(size=18),
            xanchor='center'
        ),
        # Left panel label
        dict(
            text='2004–2008: Housing Bubble<br><i>Price-driven</i>',
            xref='paper', yref='paper',
            x=0.22, y=1.08,
            showarrow=False,
            font=dict(size=13),
            xanchor='center'
        ),
        # Right panel label
        dict(
            text='2019–2024: Rate Shock Era<br><i>Both factors</i>',
            xref='paper', yref='paper',
            x=0.78, y=1.08,
            showarrow=False,
            font=dict(size=13),
            xanchor='center'
        ),
    ]
)
fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee', title_text='Monthly Payment as % of Income', row=1, col=1)
fig.show()

In [19]:
df['price_change']      = df['home_price'].pct_change()
df['rate_change']       = df['mortgage_rate'].diff()
df['rate_change_lag1']  = df['rate_change'].shift(1)
df['rate_change_lag2']  = df['rate_change'].shift(2)

df['rolling_corr']      = df['price_change'].rolling(8).corr(df['rate_change'])
df['rolling_corr_lag1'] = df['price_change'].rolling(8).corr(df['rate_change_lag1'])
df['rolling_corr_lag2'] = df['price_change'].rolling(8).corr(df['rate_change_lag2'])

fig = go.Figure()
fig.add_hline(y=0, line_color='black', line_width=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['rolling_corr'],
                         name='Same-quarter', line=dict(color='#1f77b4', width=2),
                         hovertemplate='%{x|%b %Y}<br>Corr: %{y:.2f}<extra></extra>'))
fig.add_trace(go.Scatter(x=df['date'], y=df['rolling_corr_lag1'],
                         name='1-quarter lag', line=dict(color='#ff7f0e', width=2, dash='dash'),
                         hovertemplate='%{x|%b %Y}<br>Lag-1: %{y:.2f}<extra></extra>'))
fig.add_trace(go.Scatter(x=df['date'], y=df['rolling_corr_lag2'],
                         name='2-quarter lag', line=dict(color='#2ca02c', width=2, dash='dot'),
                         hovertemplate='%{x|%b %Y}<br>Lag-2: %{y:.2f}<extra></extra>'))

fig.update_layout(
    title='Rolling Correlation: Mortgage Rate Changes vs. Home Price Changes<br><sub>8-quarter window | Negative = rate increases associated with price slowdowns</sub>',
    xaxis_title='Year', yaxis_title='Correlation Coefficient',
    yaxis=dict(range=[-1, 1]),
    height=450, hovermode='x unified',
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.show()

### Findings - Question 3

The decomposition reveals a clear structural difference between the two crises:

**2006–2008 Bubble:** Driven almost entirely by prices. Holding rates at 2004 levels, the burden still approached 28% — because prices had risen so far. Rates were not the primary culprit.

**2022–2024 Rate Shock:** Both forces hit simultaneously. Pandemic-era prices never corrected, then rates tripled in under two years — producing the highest payment burden in the full dataset.

**Rolling correlation:** The relationship between rate changes and price changes is generally negative (rates up → price growth slows). But during the 2000s bubble, prices briefly accelerated alongside rising rates as speculation overwhelmed rate sensitivity. After 2022, the negative correlation strengthened sharply. The 1-quarter lag correlation is slightly stronger in recent years, suggesting rates lead home price changes by roughly 3–6 months.

In [22]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=['Price-to-Income Ratio (×)',
                                    'Monthly Payment as % of Income',
                                    'Indexed Growth Since 1984 (1984 Q1 = 100)'])

fig.add_trace(go.Scatter(x=df['date'], y=df['price_to_income'],
                         name='Price-to-Income', line=dict(color='#1f77b4', width=2.5),
                         hovertemplate='%{x|%b %Y}<br>%{y:.2f}×<extra></extra>'),
              row=1, col=1)
fig.add_hline(y=3, line_dash='dash', line_color='red', line_width=1, row=1, col=1)

fig.add_trace(go.Scatter(x=df['date'], y=df['payment_pct_income'],
                         name='Payment % Income', line=dict(color='#d62728', width=2.5),
                         hovertemplate='%{x|%b %Y}<br>%{y:.1f}%<extra></extra>'),
              row=2, col=1)
fig.add_hline(y=28, line_dash='dash', line_color='red', line_width=1, row=2, col=1)

for col_name, label, color, dash in [
    ('price_index',    'Home Price',    '#1f77b4', 'solid'),
    ('income_index',   'Income',        '#2ca02c', 'solid'),
    ('mortgage_index', 'Mortgage Rate', '#ff7f0e', 'dot'),
]:
    fig.add_trace(go.Scatter(x=df['date'], y=df[col_name], name=label,
                             line=dict(color=color, width=2, dash=dash),
                             hovertemplate='%{x|%b %Y}<br>Index: %{y:.0f}<extra></extra>'),
                  row=3, col=1)
fig.add_hline(y=100, line_dash='dash', line_color='gray', line_width=1, row=3, col=1)


---
## Summary of Findings

| Question | Key Finding |
|----------|-------------|
| Q1: Price-to-income ratio | The ratio has been above 3× almost continuously since the early 1990s. It currently sits near an all-time high of ~6–7×, meaning the median home costs roughly twice what conventional guidelines suggest it should. |
| Q2: Monthly payment burden | The current period is the worst payment burden in the 40-year dataset — worse than the 2006–2008 bubble at its peak. The 28% threshold is being exceeded by a historically large margin. |
| Q3: Structural comparison | The 2006–2008 crisis was price-driven. The 2022–2024 crisis is a double shock — pandemic prices that never reversed, plus rates that tripled. Rolling correlation shows rates lead home price changes by roughly 1–2 quarters. |

---
## Reflection

**What surprised me:** The current affordability crisis is measurably *worse* than the 2008 housing bubble by the payment-to-income metric. I expected the bubble to be the historical worst case, but the combination of high prices *and* high rates simultaneously — something that didn't happen in 2006–2008 — makes today's market uniquely punishing.

**What was challenging:** Reconciling three different data frequencies was the biggest technical hurdle. The income series being annual created a real methodological question — forward-fill vs. interpolation. I went with interpolation, which feels more defensible, but real income doesn't change as smoothly as interpolation implies. Getting the rolling correlation right also took time; I initially ran it on raw levels rather than quarter-over-quarter changes, which produced misleadingly strong results.

**What I'd do differently with more time:**
- Add city-level Zillow data to explore geographic variation (my original research questions were all city-level)
- Include the existing home sale price series (HOSMEDUSM052N) alongside MSPUS for a more complete picture
- Break the income series down by quintile to show how affordability differs for lower-income households
- Model future scenarios: what happens to the burden if rates drop to 5%? If prices fall 20%?


## Credits

**Data:** FRED (Federal Reserve Bank of St. Louis)
- [MSPUS](https://fred.stlouisfed.org/series/MSPUS) — U.S. Census Bureau / HUD
- [MEHOINUSA646N](https://fred.stlouisfed.org/series/MEHOINUSA646N) — U.S. Census Bureau CPS
- [MORTGAGE30US](https://fred.stlouisfed.org/series/MORTGAGE30US) — Freddie Mac

**Tools:** pandas, Plotly, Google Colab

**Learning resources:**
- [pandas rolling documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html) — rolling correlation technique
- [Towards Data Science: Four Ways to Quantify Synchrony Between Time-Series Data](https://towardsdatascience.com/four-ways-to-quantify-synchrony-between-time-series-data-b99136c4a9c9)

**Assistance:** Claude (Anthropic) — used to help structure the notebook and debug code; CS-215 course materials